# Kaggle'da YOLOv8 (VisDrone) Model Eğitimi
Bu notebook, GitHub üzerindeki `visdrone-human-recognize` reponuzu Kaggle'a klonlar, gerekli kütüphaneleri kurar ve GPU üzerinde eğitimi başlatır.

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="HhMf4952IiZr1RsnKTy5")
project = rf.workspace("chris-d-dbyby").project("tinyperson")
version = project.version(5)
dataset = version.download("yolov8")

In [ ]:
import os
dataset_path = "/kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset"
print(os.listdir(dataset_path))

# Eger klasorun icinde baska klasorler varsa onlari da gorelim
try:
    print("\nDetayli klasor listesi:")
    for f in os.listdir(dataset_path):
        sub_dir = os.path.join(dataset_path, f)
        if os.path.isdir(sub_dir):
            print(f" - {f}")
except Exception as e:
    pass


## 1. Repo Klonlama ve Güncelleme
GitHub reponuzu Kaggle'ın çalışma alanına klonluyoruz.

In [ ]:
import os

# AŞAĞIDAKİ URL'Yİ KENDİ GITHUB REPO LİNKİNİZ İLE DEĞİŞTİRİN!
GITHUB_REPO_URL = "https://github.com/emremustu/visdrone-human-recognize.git"
PROJECT_DIR = "visdrone-human-recognize"

%cd /kaggle/working
if not os.path.exists(PROJECT_DIR):
    print("Repo klonlanıyor...")
    !git clone {GITHUB_REPO_URL}
else:
    print("Repo zaten var, güncelleniyor...")
    %cd {PROJECT_DIR}
    !git pull
    %cd /kaggle/working

%cd {PROJECT_DIR}

## 2. Bağımlılıkları Kurma
Projenizdeki `requirements.txt` dosyasını kullanarak bağımlılıkları yüklüyoruz.

In [ ]:
!pip install -r requirements.txt
!pip install ultralytics clearml  # Ultralytics (YOLO) kesinlikle yüklü olmalı

## 3. GPU ve Veri Kontrolü
Kaggle GPU'sunun aktif olup olmadığına ve veri setinin eklenip eklenmediğine bakıyoruz.

In [ ]:
import torch
print("CUDA Kullanılabilir mi?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Modeli:", torch.cuda.get_device_name(0))
else:
    print("UYARI: GPU AKTİF DEĞİL! Sağ üstteki 'Settings' > 'Accelerator' altından GPU T4 x2 Seçin.")

import os
print("\n--- /kaggle/input (VERİ SETLERİ) İÇİNDEKİ DOSYALAR ---")
# Eğer bu liste boş geliyorsa sağ üstten 'Add Data' diyip Kaggle veri setini notebook'a eklemelisiniz.
if os.path.exists("/kaggle/input"):
    print(os.listdir("/kaggle/input"))
else:
    print("/kaggle/input klasörü bulunamadı.")

## 4. Eğitimi Başlatma
Projedeki `train.py` (veya sizin başlangıç script'iniz) üzerinden eğitimi başlatıyoruz.

In [ ]:
import yaml
import os

# 1. Projendeki orjinal visdrone.yaml'ı açalım
config_path = "/kaggle/working/visdrone-human-recognize/configs/visdrone.yaml"

# 2. İçindeki 'path' satırını Kaggle Input (senin indirdiğin veri setinin klasörü) yoluna göre güncelleyelim.
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# Kaggle dataset yolu
config["path"] = "/kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset"

# 3. Yolu güncelleyip kaydedelim
with open(config_path, "w") as f:
    yaml.dump(config, f)

print(f"✅ config yaml başarıyla Kaggle datasetine bağlandı: {config['path']}")

# 4. Eğitimi kendi güncellediğimiz yaml ile başlatalım
!python train.py --data {config_path} --epochs 50 --batch 16 --imgsz 640


## 5. Eğitilmiş Modelleri Zipleyerek İndirmeye Hazır Hale Getirme (İsteğe Bağlı)
Kaggle ekranında Output kısmından tek tek dosya indirmek yerine tüm `runs` klasörünü zip yapabilirsiniz.

In [ ]:
import shutil

runs_folder = "runs"
output_zip = "/kaggle/working/egitim_sonuclari"

if os.path.exists(runs_folder):
    shutil.make_archive(output_zip, 'zip', runs_folder)
    print(f"{runs_folder} klasörü {output_zip}.zip olarak sıkıştırıldı. Output sekmesinden indirebilirsiniz.")
else:
    print(f"{runs_folder} klasörü bulunamadı, henüz eğitim tamamlanmamış olabilir.")